In [3]:
from pathlib import Path
import pandas as pd

interim = Path("datos/interim")
if not interim.is_dir():
    interim = Path("../datos/interim")

participaciones = pd.read_csv(
    interim / "resultados_limpios_local.csv",
    dtype={"dorsal": "string", "nombre_completo": "string"},
)
inscritos = pd.read_csv(
    interim / "inscritos_circuito_interim.csv",
    dtype="string",
    keep_default_na=False,
)

def normalizar_nombre(columna):
    return (
        columna.fillna("")
        .str.normalize("NFKD")
        .str.replace(r"[\u0300-\u036f]", "", regex=True)
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

for tabla in (participaciones, inscritos):
    tabla["nombre_normalizado"] = normalizar_nombre(tabla["nombre_completo"])
    tabla["dorsal"] = tabla["dorsal"].fillna("").str.strip()

print("Participaciones:", len(participaciones))
print("Registros en el listado de inscritos:", len(inscritos))

con_nombre_y_dorsal = participaciones.loc[
    participaciones["nombre_normalizado"].ne("") & participaciones["dorsal"].ne("")
]
nombres_por_dorsal = con_nombre_y_dorsal.groupby("dorsal")["nombre_normalizado"].nunique()
print("Dorsales asociados a varios nombres:", int(nombres_por_dorsal.gt(1).sum()))

nombre_repetido = participaciones["nombre_normalizado"].ne("") & (
    participaciones.duplicated(["id_carrera", "nombre_normalizado"], keep=False)
)
print("Filas con nombre repetido dentro de una carrera:", int(nombre_repetido.sum()))

Participaciones: 5390
Registros en el listado de inscritos: 2345
Dorsales asociados a varios nombres: 315
Filas con nombre repetido dentro de una carrera: 21


In [4]:
# Una pareja exacta usa los dos campos: nombre normalizado y dorsal.
campos = ["nombre_normalizado", "dorsal"]
padron = inscritos.loc[
    inscritos["nombre_normalizado"].ne("") & inscritos["dorsal"].ne(""),
    campos,
].drop_duplicates()

parejas_padron = pd.MultiIndex.from_frame(padron[campos])
parejas_resultados = pd.MultiIndex.from_frame(participaciones[campos])
pareja_exacta = pd.Series(
    parejas_resultados.isin(parejas_padron),
    index=participaciones.index,
)

dorsales_por_nombre = padron.groupby("nombre_normalizado")["dorsal"].nunique()
candidatos = (
    participaciones["nombre_normalizado"]
    .map(dorsales_por_nombre)
    .fillna(0)
    .astype(int)
)

# Si el nombre se repite en alguna carrera, señalamos también
# sus apariciones en las demás para evitar unirlas por nombre.
nombres_dudosos = participaciones.loc[nombre_repetido, "nombre_normalizado"]
nombre_dudoso = participaciones["nombre_normalizado"].isin(nombres_dudosos)

auditoria = pd.Series(
    "sin coincidencia en el listado", index=participaciones.index
)
auditoria.loc[candidatos.eq(1)] = "un candidato por nombre, distinto dorsal"
auditoria.loc[pareja_exacta] = "coinciden nombre y dorsal"
auditoria.loc[nombre_dudoso | candidatos.gt(1)] = "nombre ambiguo, revisar"
auditoria.loc[participaciones["nombre_normalizado"].eq("")] = "sin nombre"

assert len(auditoria) == len(participaciones)
print(auditoria.value_counts().to_string())
print("Filas comprobadas:", len(auditoria))

coinciden nombre y dorsal                   3833
sin coincidencia en el listado              1458
un candidato por nombre, distinto dorsal      72
nombre ambiguo, revisar                       27
Filas comprobadas: 5390


In [5]:
cruce = participaciones.copy()
cruce["resultado_cruce"] = auditoria

coincidencia_exacta = auditoria.eq("coinciden nombre y dorsal")
sin_coincidencia = auditoria.eq("sin coincidencia en el listado")
pendiente = ~(coincidencia_exacta | sin_coincidencia)

# Para los casos claros, conservamos una clave estable entre carreras.
cruce["clave_persona"] = "nombre:" + cruce["nombre_normalizado"]

cruce.loc[coincidencia_exacta, "clave_persona"] = (
    "padron:" + cruce.loc[coincidencia_exacta, "dorsal"]
    + ":" + cruce.loc[coincidencia_exacta, "nombre_normalizado"]
)

# Para los casos dudosos, mantenemos cada participación separada.
cruce.loc[pendiente, "clave_persona"] = (
    "pendiente:" + cruce.loc[pendiente, "id_carrera"]
    + ":" + cruce.loc[pendiente, "dorsal"]
)

# Una misma clave debe aparecer como máximo una vez por carrera.
assert len(cruce) == 5390
assert not cruce.duplicated(["id_carrera", "clave_persona"]).any(), (
    "Hay claves repetidas en una carrera. Revisa la auditoría antes de continuar."
)

columnas_auditoria = [
    "id_carrera", "dorsal", "nombre_completo", "resultado_cruce"
]
cruce.loc[pendiente, columnas_auditoria].to_csv(
    interim / "auditoria_enlace_local.csv", index=False
)

print("Coinciden nombre y dorsal:", int(coincidencia_exacta.sum()))
print("Sin coincidencia en el listado:", int(sin_coincidencia.sum()))
print("Pendientes de revisión:", int(pendiente.sum()))
print("Filas conservadas:", len(cruce))

Coinciden nombre y dorsal: 3833
Sin coincidencia en el listado: 1458
Pendientes de revisión: 99
Filas conservadas: 5390


In [6]:
import hashlib
import hmac

raiz = interim.resolve().parents[1]
archivo_env = raiz / ".env"
assert archivo_env.is_file(), "Falta el archivo .env en la raíz del repositorio."

variables = dict(
    linea.split("=", 1)
    for linea in archivo_env.read_text(encoding="utf-8").splitlines()
    if "=" in linea and not linea.lstrip().startswith("#")
)
clave = variables.get("CORUNA_HMAC_KEY", "").strip().encode("utf-8")
assert clave, "Falta CORUNA_HMAC_KEY en .env."

def generar_id(clave_persona):
    return hmac.new(
        clave,
        clave_persona.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()[:12]

cruce["persoa_id"] = cruce["clave_persona"].map(generar_id)

# Persona 1 confirmó que el listado contiene las inscripciones válidas.
cruce["tipo_inscripcion"] = "no_resuelto"
cruce.loc[coincidencia_exacta, "tipo_inscripcion"] = "inscrito_circuito"
cruce.loc[sin_coincidencia, "tipo_inscripcion"] = "por_prueba"

participacions = cruce[[
    "persoa_id", "id_carrera", "tipo_inscripcion", "resultado_cruce",
    "sexo", "categoria", "puesto_general", "puesto_por_sexo",
    "puesto_por_categoria", "tiempo_segundos", "estado_tiempo",
]].copy()

carreiras = pd.read_csv(interim / "carreiras_base.csv")
carreiras["fecha"] = pd.to_datetime(carreiras["fecha"], format="%Y-%m-%d")
carreiras = carreiras.sort_values("fecha").reset_index(drop=True)
carreiras["orde"] = carreiras.index + 1

assert len(participacions) == len(participaciones) == 5390
assert cruce["persoa_id"].nunique() == cruce["clave_persona"].nunique()
assert not participacions.duplicated(["persoa_id", "id_carrera"]).any()
assert len(carreiras) == 6 and carreiras["id_carrera"].is_unique
assert participacions["id_carrera"].isin(carreiras["id_carrera"]).all()

cruce[["persoa_id", "id_carrera", "nombre_normalizado", "resultado_cruce"]].to_csv(
    interim / "enlace_personas_local.csv", index=False
)

clean = raiz / "datos" / "clean"
clean.mkdir(parents=True, exist_ok=True)
participacions.to_csv(clean / "participacions.csv", index=False)
carreiras.to_csv(clean / "carreiras.csv", index=False)

print("Participaciones exportadas:", len(participacions))
print("Carreras exportadas:", len(carreiras))
print("Pendientes de revisión de identidad:", int(pendiente.sum()))
print(participacions["tipo_inscripcion"].value_counts().to_string())

Participaciones exportadas: 5390
Carreras exportadas: 6
Pendientes de revisión de identidad: 99
tipo_inscripcion
inscrito_circuito    3833
por_prueba           1458
no_resuelto            99
